# Syntax	                        Purpose

- @dp.table	Decorator that registers a Python function as a pipeline table
- @dp.temporary_view	Decorator that registers a function as a temporary view
- @dp.expect_all_or_drop(rules)	Decorator that attaches data-quality expectations
- dp.create_streaming_table(...)	Direct function call
- dp.create_auto_cdc_flow(...)	Direct function call

### Spark Declaritive Pipeline

In [0]:
from pyspark import pipelines as dp
from pyspark.sql.functions import *

# Data quality expectations
rules ={"rule1" : "product_id IS NOT NULL",
        "rule2" : "product_name IS NOT NULL"}


# Staging streaming table
@dp.table(name="DimProducts_stg") # if not give table name inside then the function name will be table name
@dp.expect_all_or_drop(rules)

def DimProducts_stg():
    df = spark.readStream.table("DEV.SILVER.products_silver")
    return df



### Streaming Views

In [0]:
@dp.temporary_view(name="Dim_Product_view")

def Dim_Product_view():
    df= spark.readStream.table("DimProducts_stg")  #LIVE key word is used to read pipeline at a time(SDP)
    return df

### DimProducts

In [0]:
dp.create_streaming_table(
    name="DimProducts"
)


In [0]:
dp.create_auto_cdc_flow(
    target="DimProducts",
    source="Dim_Product_view",
    keys=["product_id"],
    sequence_by="updated_at",
    stored_as_scd_type=2
)